# Smart Traffic Light DQN Training (Kaggle)

Trains the DQN agent from `truongNgn/smart-traffic-light-system` against the SUMO/TraCI Gymnasium env, using Kaggle's free GPU.

**Before running:** in the notebook settings (right sidebar), set **Accelerator = GPU T4 x2** and **Internet = On** (needed for `apt-get install sumo` and `git clone`).

See `docs/training_on_kaggle.md` in the repo for the full writeup this notebook follows.

## 1. Install SUMO and clone the repo

In [ ]:
!apt-get update -qq && apt-get install -y -qq sumo sumo-tools sumo-doc

In [ ]:
!git clone https://github.com/truongNgn/smart-traffic-light-system.git
%cd smart-traffic-light-system

In [ ]:
# torch is already preinstalled on Kaggle's GPU image - don't reinstall it
!pip install -q pydantic pydantic-settings structlog traci sumolib gymnasium numpy

In [ ]:
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"

!sumo --version

## 2. Confirm GPU is visible to PyTorch

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. Build the SUMO network and generate demand

In [ ]:
!python -m simulation.net.build_net
!python -m simulation.net.generate_routes --duration 3600 --seed 42

## 4. Quick pipeline smoke test (optional but recommended)

Runs 2 tiny episodes end-to-end before committing to a long training run - catches setup problems in seconds instead of hours.

In [ ]:
from rl.train.config import TrainingConfig
from rl.train.train import train

smoke_cfg = TrainingConfig(
    num_episodes=2,
    episode_duration_s=60,
    checkpoint_dir="/kaggle/working/smoke_checkpoints",
    min_replay_size=8,
    batch_size=4,
)
_ = train(smoke_cfg)
print("Smoke test OK")

## 5. Train

Adjust `num_episodes` / `episode_duration_s` / `batch_size` as needed. GPU can afford a larger batch than the CPU default (64).

In [ ]:
from rl.train.config import TrainingConfig
from rl.train.train import train

cfg = TrainingConfig(
    num_episodes=500,
    episode_duration_s=3600,
    checkpoint_dir="/kaggle/working/checkpoints",
    batch_size=128,
)
agent = train(cfg)

## 6. Resuming (only needed if a session got cut off)

Kaggle sessions get killed after ~9-12 hours. `/kaggle/working/` output persists between sessions, so if training didn't finish, start a new session and continue from the last checkpoint instead of restarting from scratch.

In [ ]:
# from rl.train.config import TrainingConfig
# from rl.train.train import train
#
# resumed_cfg = TrainingConfig(
#     num_episodes=1000,
#     episode_duration_s=3600,
#     checkpoint_dir="/kaggle/working/checkpoints",
#     resume_from="/kaggle/working/checkpoints/dqn_final.pt",
# )
# agent = train(resumed_cfg)

## 7. Download the trained model

Checkpoints are written to `/kaggle/working/checkpoints/` (`dqn_best.pt`, `dqn_final.pt`, and a periodic `dqn_episode_N.pt`). They show up in the notebook's **Output** tab for download after the session ends, or grab them directly from this cell.

Bring `dqn_best.pt` back to the local repo's `checkpoints/` directory (already gitignored) to use with Stage 5's evaluation/benchmark scripts:

```python
from rl.agent.dqn_agent import DQNAgent
from rl.train.checkpoint import load_checkpoint

agent = DQNAgent()
load_checkpoint("checkpoints/dqn_best.pt", agent)
```

In [ ]:
!ls -la /kaggle/working/checkpoints